# Heart Disease Prediction — Data Preparation & EDA

## 1.1 Problem Statement

This is a binary classification problem: given a patient's clinical 
measurements, predict whether they currently have heart disease (1) or 
not (0). Because false negatives (missing a real case) carry greater 
real-world risk than false positives in a medical context, model 
evaluation will prioritize recall alongside accuracy.

**Dataset:** [Heart Failure Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) 
(918 patients, 11 clinical features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../data/heart.csv')
df.head()

In [ ]:
df.shape
df.info()
df.isnull().sum()

In [ ]:
df.describe()

### 1.2 Data Cleaning
**Cholesterol and RestingBP contain disguised missing values encoded as 0,which is not physiologically possible. isnull() does not catch these.**

In [ ]:
print("Zero values — Cholesterol:", (df['Cholesterol'] == 0).sum())
print("Zero values — RestingBP:", (df['RestingBP'] == 0).sum())

In [ ]:
# Checked whether missing Cholesterol values were randomly distributed 
# or correlated with the target — found a ~6x disparity between classes 
# (29.9% of HeartDisease=1 patients affected vs. 4.9% of HeartDisease=0), 
# indicating Missing Not At Random (MNAR) rather than random noise.
df.groupby('HeartDisease')['Cholesterol'].apply(lambda x: (x == 0).sum())

In [ ]:
# Cholesterol: flag missingness before imputing (MNAR pattern above means 
# missingness itself may carry signal), then impute using the median of 
# valid readings only.
df['Cholesterol_missing'] = (df['Cholesterol'] == 0).astype(int)
median_cholesterol = df.loc[df['Cholesterol'] != 0, 'Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, median_cholesterol)

# RestingBP: single affected row (0.1% of data) — simple median imputation 
# is proportional to the scale of the issue; no indicator column needed.
median_bp = df.loc[df['RestingBP'] != 0, 'RestingBP'].median()
df['RestingBP'] = df['RestingBP'].replace(0, median_bp)

**Known limitation:** Cholesterol imputation may dilute the true 
relationship between cholesterol and heart disease, since 88% of the 
imputed values belong to positive cases. Model-based imputation would 
be a stronger approach in a future iteration.

## 1.3 Exploratory Data Analysis

**Visualizing the Data**

**What we want to do here is that:**
Before building any model, we need to understand the shape of our target 
variable and how our features relate to it. This step helps us form 
early intuitions about what the model might learn, and helps us catch 
any surprises in the data before we commit to a modeling approach.

**Class Distribution**

Since this is a binary classification problem, the first thing we check 
is how balanced our two classes are — patients with heart disease (1) 
vs. patients without (0).

**Why this matters:** If one class dominates (e.g., 95% healthy, 5% sick), 
accuracy alone becomes a misleading metric — a model could achieve 95% 
accuracy by just always predicting "healthy" without learning anything 
useful. Understanding this now shapes how we evaluate our model later.

In [ ]:
# Class balance — confirms a reasonably balanced target (~55%/45%), 
# supporting accuracy as a meaningful evaluation metric.
df['HeartDisease'].value_counts(normalize=True)

In [ ]:
sns.countplot(data=df, x='HeartDisease')
plt.title('Class Distribution: Heart Disease')
plt.xlabel('0 = No Disease, 1 = Has Disease')
plt.show()

In [ ]:
# Numeric feature distributions by class
for col in ['Age', 'Cholesterol', 'MaxHR']:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x='HeartDisease', y=col)
    plt.title(f'{col} by Heart Disease Status')
    plt.show()

**Male patients in this dataset are considerably more likely to have heart disease, while female patients are considerably more likely to NOT have it.**

In [ ]:
df["Sex"].value_counts()

**This dataset shows that it's heavily skewed toward male patients — roughly 4x more men than women.**

The model will see far more examples of male patients, in far more varied situations, than female patients. This means whatever pattern it learns about "Sex" is disproportionately driven by data from men.

**This is a subgroup performance gap**

In [ ]:
# Categorical feature distributions by class
for col in ['ChestPainType', 'Sex', 'ExerciseAngina', 'ST_Slope']:
    plt.figure(figsize=(7,4))
    sns.countplot(data=df, x=col, hue='HeartDisease')
    plt.title(f'{col} vs Heart Disease')
    plt.show()

**Key EDA findings:**
- **Strong signal:** MaxHR, ChestPainType (ASY), ExerciseAngina, 
  ST_Slope (Flat) — all show clear separation between classes and 
  align with established clinical patterns.
- **Weak signal:** Cholesterol shows minimal separation, likely 
  affected by imputation (see limitation above).
- **Class imbalance within a feature:** Sex is 79% male / 21% female. 
  While Sex shows a strong association with the target, this imbalance 
  means the model's understanding of female patients is based on a 
  smaller, less representative sample — flagged for subgroup evaluation 
  during model assessment.

## 1.4 Encoding Categorical Features

Before we can train a logistic regression model, every input must be 
numeric.

We use two different encoding strategies depending on the number of 
categories in each column:

- **Label Encoding** for binary categories (Sex, ExerciseAngina) — 
  simply mapping to 0/1, since there's no risk of implying a false order.
- **One-Hot Encoding** for categories with 3+ options (ChestPainType, 
  RestingECG, ST_Slope) — creating separate binary columns per category, 
  since assigning arbitrary numbers (0,1,2,3) would falsely imply ranking 
  that doesn't exist between them.

 with 
`drop_first=True` to avoid redundant, perfectly collinear columns.

In [ ]:
df['Sex'] = df['Sex'].map({'M': 1, 'F': 0})
df['ExerciseAngina'] = df['ExerciseAngina'].map({'Y': 1, 'N': 0})

df = pd.get_dummies(df, columns=['ChestPainType', 'RestingECG', 'ST_Slope'], 
                     drop_first=True)

## 1.5 Train/Test Split

Data is split 80/20, stratified on the target to preserve class balance 
across both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

In [ ]:
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining set class balance:")
print(y_train.value_counts(normalize=True))
print("\nTest set class balance:")
print(y_test.value_counts(normalize=True))

## 1.6 Feature Scaling

Logistic regression's gradient descent is sensitive to feature scale — 
features with larger numeric ranges (e.g., Cholesterol: 0-600+) produce 
disproportionately large gradient updates compared to smaller-scale 
features (e.g., Sex: 0/1), causing inefficient, uneven convergence.

We standardize numeric features to mean=0, std=1 using scikit-learn's 
StandardScaler.

**Critical rule:** The scaler is fit ONLY on training data, then used 
to transform both train and test sets. Fitting on combined data would 
leak test set statistics into training — a form of data leakage that 
makes evaluation results artificially optimistic.## 1.6 Feature Scaling

Logistic regression's gradient descent is sensitive to feature scale — 
features with larger numeric ranges (e.g., Cholesterol: 0-600+) produce 
disproportionately large gradient updates compared to smaller-scale 
features (e.g., Sex: 0/1), causing inefficient, uneven convergence.

We standardize numeric features to mean=0, std=1 using scikit-learn's 
StandardScaler.

**Critical rule:** The scaler is fit ONLY on training data, then used 
to transform both train and test sets. Fitting on combined data would 
leak test set statistics into training — a form of data leakage that 
makes evaluation results artificially optimistic.

In [ ]:
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

we Identified which columns actually need scaling.
Binary/one-hot columns (already 0/1) don't need scaling

In [ ]:
# Fit the scaler using ONLY training data statistics (mean, std)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

In [ ]:
# Transform test data using the SAME scaler (fitted on train only) — 
# never re-fit on test data
X_test_scaled = X_test.copy()
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
# scaled training data should have mean ≈ 0, std ≈ 1
print(X_train_scaled[numeric_cols].describe().loc[['mean', 'std']])

## 1.7: Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Initialize Logistic Regression with reproducible training
# Increased iteration limit to ensure convergence on encoded feature set
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train  )

In [ ]:
print("model Training complete.")
print('Number of iterations:', model.n_iter_)

## 1.8: Generating Predictions

In [ ]:
# Generate class predictions and probability estimates on held-out test set
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

## 1.9: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score, precision_score, f1_score

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"test Accuracy: {accuracy:.4f}")
cm = confusion_matrix(y_test, y_pred)
cm

In [62]:
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recal: {recall:.4f}")
print(f"f1-score: {f1:.4f}")

Precision: 0.8952
Recal: 0.9216
f1-score: 0.9082
